In [1]:
!pip install deap yfinance openpyxl plotly --quiet

# Módulo 2: Algoritmo Genético NSGA-II

Este notebook implementa la optimización multiobjetivo de portafolios utilizando el algoritmo genético NSGA-II (a través de la librería DEAP). El objetivo es encontrar el conjunto de portafolios que representan el mejor compromiso entre maximizar el retorno esperado y minimizar el riesgo (volatilidad), conformando el Frente de Pareto. Finalmente, se selecciona el portafolio con el mejor Sharpe Ratio dentro del frente, se realiza su simulación de riqueza histórica y drawdowns, y se exportan los resultados.

In [2]:
import os
import json
import io
import time
import random
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from deap import base, creator, tools, algorithms

# Crear carpeta de salidas si no existe
os.makedirs("salidas", exist_ok=True)

# Parámetros globales
TICKERS = ['ABX.TO', 'BHP', 'BVN', 'FSM', 'VOLCABC1.LM']
FECHA_INICIO = '2015-01-01'
FECHA_FIN = '2024-12-31'
POBLACION = 100
GENERACIONES = 80
CAPITAL_INICIAL = 100000
RF = 0.0
DIAS_ANIO = 252

## 1. Carga y Limpieza de Datos

Descargamos los precios históricos de cierre (`Close`) de los activos seleccionados desde Yahoo Finance y calculamos los retornos diarios logarítmicos, el vector de retornos anualizados ($\mu$) y la matriz de covarianza anualizada ($\Sigma$).

In [3]:
print(f"Descargando datos para tickers: {TICKERS}")
descarga = yf.download(TICKERS, start=FECHA_INICIO, end=FECHA_FIN, progress=False)

if isinstance(descarga, pd.DataFrame) and not descarga.empty:
    if 'Close' in descarga.columns:
        precios = descarga['Close']
    else:
        precios = descarga
else:
    raise ValueError("Error al descargar los precios históricos.")

precios = precios.ffill().bfill()
TICKERS_VALIDOS = list(precios.columns)
N = len(TICKERS_VALIDOS)

retornos = pd.DataFrame(np.log(precios / precios.shift(1))).dropna()
mu = retornos.mean().to_numpy(dtype=float) * DIAS_ANIO
Sigma = retornos.cov().to_numpy(dtype=float) * DIAS_ANIO

print(f"Datos descargados con éxito. Activos válidos: {TICKERS_VALIDOS}")
precios.head()

Descargando datos para tickers: ['ABX.TO', 'BHP', 'BVN', 'FSM', 'VOLCABC1.LM']


Datos descargados con éxito. Activos válidos: ['ABX.TO', 'BHP', 'BVN', 'FSM', 'VOLCABC1.LM']


Ticker,ABX.TO,BHP,BVN,FSM,VOLCABC1.LM
Date,,,,,
2015-01-02,10.361854,20.644577,8.838104,4.68,0.663575
2015-01-05,10.321572,19.910683,9.368209,4.91,0.654358
2015-01-06,10.724441,19.784750,9.733797,5.09,0.654358
2015-01-07,10.595522,19.971476,9.624122,4.98,0.645142
2015-01-08,10.305456,20.271116,9.395627,4.89,0.645142


## 2. Configuración de DEAP para NSGA-II

Configuramos las clases de aptitud (Fitness) y de individuo para el algoritmo genético. En este caso, el objetivo es doble: maximizar retorno (peso 1.0) y minimizar riesgo (peso -1.0). También definimos las funciones personalizadas para evaluación, cruzamiento y mutación asegurando que los pesos de los portafolios sumen 1.0 y estén acotados en $[0, 1]$.

In [4]:
# Registrar tipos en creator
if not hasattr(creator, "FitnessMulti"):
    creator.create("FitnessMulti", base.Fitness, weights=(1.0, -1.0))
if not hasattr(creator, "Individual"):
    creator.create("Individual", list, fitness=creator.FitnessMulti)

def crear_individuo(n):
    pesos = np.random.random(n)
    pesos /= np.sum(pesos)
    return creator.Individual(pesos.tolist())

def evaluar_portafolio(ind, mu_vec, sigma_mat):
    w = np.array(ind, dtype=float)
    if np.sum(w) == 0:
        return -1e9, 1e9
    w /= np.sum(w)
    retorno = float(np.dot(w, mu_vec))
    riesgo = float(np.sqrt(w @ sigma_mat @ w))
    return retorno, riesgo

def mutacion_portafolio(ind, indpb):
    for i in range(len(ind)):
        if random.random() < indpb:
            ind[i] += random.gauss(0, 0.1)
            ind[i] = max(0.001, ind[i])
    suma = sum(ind)
    for i in range(len(ind)):
        ind[i] /= suma
    return ind,

def cruce_portafolio(ind1, ind2):
    for i in range(len(ind1)):
        if random.random() < 0.5:
            ind1[i], ind2[i] = ind2[i], ind1[i]
    sum1, sum2 = sum(ind1), sum(ind2)
    for i in range(len(ind1)):
        ind1[i] /= sum1
        ind2[i] /= sum2
    return ind1, ind2

toolbox = base.Toolbox()
toolbox.register("individual", crear_individuo, N)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluar_portafolio, mu_vec=mu, sigma_mat=Sigma)
toolbox.register("mate", cruce_portafolio)
toolbox.register("mutate", mutacion_portafolio, indpb=0.2)
toolbox.register("select", tools.selNSGA2)

## 3. Ejecución del Algoritmo Genético NSGA-II

Corremos la evolución genética para la población configurada durante el número de generaciones especificado. Al final, extraemos los individuos que pertenecen al primer frente de Pareto no dominado.

In [5]:
random.seed(42)
np.random.seed(42)

# Inicializar población
pop = toolbox.population(n=POBLACION)

# Evaluar población inicial
fitnesses = list(map(toolbox.evaluate, pop))
for ind, fit in zip(pop, fitnesses):
    ind.fitness.values = fit

pop = toolbox.select(pop, len(pop))

print(f"Ejecutando NSGA-II: población = {POBLACION}, generaciones = {GENERACIONES}...")

# Bucle generacional
for gen in range(GENERACIONES):
    offspring = tools.selTournamentDCD(pop, len(pop))
    offspring = [toolbox.clone(ind) for ind in offspring]
    
    # Cruce
    for ind1, ind2 in zip(offspring[::2], offspring[1::2]):
        if random.random() < 0.8:
            toolbox.mate(ind1, ind2)
            del ind1.fitness.values, ind2.fitness.values
            
    # Mutación
    for ind in offspring:
        if random.random() < 0.2:
            toolbox.mutate(ind)
            del ind.fitness.values
            
    # Evaluar individuos inválidos
    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit
        
    # Selección de supervivientes (P + Q)
    pop = toolbox.select(pop + offspring, POBLACION)

# Extraer Frente de Pareto Final
frente_pareto = tools.sortNondominated(pop, len(pop), first_front_only=True)[0]
print(f"Frente de Pareto extraído con éxito. Número de portafolios óptimos: {len(frente_pareto)}")

Ejecutando NSGA-II: población = 100, generaciones = 80...


Frente de Pareto extraído con éxito. Número de portafolios óptimos: 100


## 4. Selección del Mejor Individuo (Máximo Sharpe)

Extraemos los valores de riesgo y retorno de los individuos del frente de Pareto final, calculamos su Sharpe Ratio y seleccionamos el mejor.

In [6]:
riesgos = [ind.fitness.values[1] for ind in frente_pareto]
retornos_pareto = [ind.fitness.values[0] for ind in frente_pareto]
sharpes = [(ret - RF) / rsk if rsk > 0 else 0 for ret, rsk in zip(retornos_pareto, riesgos)]

idx_mejor_sharpe = np.argmax(sharpes)
mejor_ind = frente_pareto[idx_mejor_sharpe]
mejor_pesos = np.array(mejor_ind) / np.sum(mejor_ind)

ret_f = float(retornos_pareto[idx_mejor_sharpe])
rsk_f = float(riesgos[idx_mejor_sharpe])
sharpe_f = float(sharpes[idx_mejor_sharpe])

print("--- MEJOR PORTAFOLIO ENCONTRADO POR NSGA-II ---")
print(f"Sharpe Ratio: {sharpe_f:.4f}")
print(f"Retorno Esperado Anualizado: {ret_f*100:.2f}%")
print(f"Volatilidad Anualizada (Riesgo): {rsk_f*100:.2f}%")
print("Pesos del portafolio óptimo evolutivo:")
for t, w in zip(TICKERS_VALIDOS, mejor_pesos):
    print(f"  {t}: {w*100:.2f}%")

--- MEJOR PORTAFOLIO ENCONTRADO POR NSGA-II ---
Sharpe Ratio: 0.2777
Retorno Esperado Anualizado: 7.75%
Volatilidad Anualizada (Riesgo): 27.89%
Pesos del portafolio óptimo evolutivo:
  ABX.TO: 35.94%
  BHP: 64.00%
  BVN: 0.02%
  FSM: 0.02%
  VOLCABC1.LM: 0.01%


## 5. Visualizaciones de Resultados

Trazamos el Frente de Pareto final y la composición del portafolio óptimo evolutivo (Dona), junto con el desempeño de los activos individuales (Barras).

In [7]:
# 1. Frente de Pareto con Plotly
fig_pareto = go.Figure()
fig_pareto.add_trace(go.Scatter(
    x=riesgos, y=retornos_pareto, mode='markers',
    name='Frente de Pareto Final',
    marker=dict(color='blue', size=8, opacity=0.7)
))
fig_pareto.add_trace(go.Scatter(
    x=[rsk_f], y=[ret_f], mode='markers',
    name='Mejor Sharpe Evolutivo',
    marker=dict(color='red', symbol='star', size=16)
))
fig_pareto.update_layout(
    title='Frente de Pareto Obtenido por NSGA-II',
    xaxis_title='Riesgo (Volatilidad)',
    yaxis_title='Retorno Esperado',
    template='plotly_dark'
)
fig_pareto.show()

# 2. Composición de Pesos (Dona)
df_pesos_ga = pd.DataFrame({'Ticker': TICKERS_VALIDOS, 'Peso (%)': (mejor_pesos * 100).round(2)})
df_pesos_ga_filtrado = df_pesos_ga[df_pesos_ga['Peso (%)'] > 0.01]
fig_pie_ga = px.pie(
    df_pesos_ga_filtrado, 
    values='Peso (%)', 
    names='Ticker',
    title='Composición del Portafolio Óptimo (Pesos %)', 
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig_pie_ga.update_layout(template='plotly_dark')
fig_pie_ga.show()

# 3. Desempeño Individual de los Activos
df_activos = pd.DataFrame({
    'Ticker': TICKERS_VALIDOS,
    'Retorno Anualizado (%)': (mu * 100).round(2),
    'Volatilidad Anualizada (%)': (np.sqrt(np.diag(Sigma)) * 100).round(2)
})
df_long = df_activos.melt(id_vars='Ticker', value_vars=['Retorno Anualizado (%)', 'Volatilidad Anualizada (%)'],
                          var_name='Métrica', value_name='Valor (%)')
fig_activos = px.bar(
    df_long,
    x='Ticker',
    y='Valor (%)',
    color='Métrica',
    barmode='group',
    title='Desempeño Individual de los Activos',
    color_discrete_sequence=['#2ecc71', '#e74c3c']
)
fig_activos.update_layout(template='plotly_dark')
fig_activos.show()

# 4. Matriz de Correlación de Activos
correlaciones = retornos.corr()
fig_heatmap = px.imshow(
    correlaciones,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu",
    zmin=-1, zmax=1,
    title="Matriz de Correlación de Activos"
)
fig_heatmap.update_layout(template='plotly_dark')
fig_heatmap.show()

## 6. Simulación de Riqueza Histórica (Backtesting) y Caídas (Drawdowns)

Realizamos la simulación histórica del portafolio óptimo seleccionado en el Frente de Pareto, considerando rebalanceo Mensual y costos de transacción (0.10%), y calculamos las caídas máximas (Drawdowns).

In [8]:
frecuencia_sel = "Mensual"
codigo_freq = "ME"
LAMBDA_TC = 0.0010

precios_sim = precios.resample(codigo_freq).last()
retornos_sim = precios_sim.pct_change().dropna()
fechas_sim = [precios_sim.index[0]] + list(retornos_sim.index)

# Motor unificado (idéntico al de M4)
def backtest_ga(retornos_df, w_objetivo, capital_inicial, lambda_tc=0.0):
    R = retornos_df.to_numpy(dtype=float)
    T_len = R.shape[0]
    w_obj = np.asarray(w_objetivo, dtype=float)
    riqueza = np.zeros(T_len + 1)
    riqueza[0] = capital_inicial
    w_actual = w_obj.copy()
    
    for t in range(T_len):
        ret = R[t]
        if t > 0:
            delta = np.sum(np.abs(w_actual - w_obj))
            tc = lambda_tc * delta
            cap_post = riqueza[t] * (1 - tc)
            ret_port = float(np.dot(w_obj, ret))
            riqueza[t + 1] = cap_post * (1 + ret_port)
            w_actual = w_obj * (1 + ret) / (1 + ret_port)
        else:
            ret_port = float(np.dot(w_actual, ret))
            riqueza[t + 1] = riqueza[t] * (1 + ret_port)
            w_actual = w_actual * (1 + ret) / (1 + ret_port)
    return riqueza

riqueza_nsga = backtest_ga(retornos_sim, mejor_pesos, CAPITAL_INICIAL, LAMBDA_TC)

# Gráfico de evolución de riqueza
fig_sim = go.Figure()
fig_sim.add_trace(go.Scatter(x=fechas_sim, y=riqueza_nsga, mode='lines', name=f'NSGA-II Rebalanceado (${riqueza_nsga[-1]:,.2f})', line=dict(color='blue')))
fig_sim.update_layout(title=f'Crecimiento de Estrategia NSGA-II (Rebalanceo {frecuencia_sel})', xaxis_title='Fecha', yaxis_title='Capital (USD)', template='plotly_dark')
fig_sim.show()

# Calcular Drawdowns
cum_max = np.maximum.accumulate(riqueza_nsga)
with np.errstate(divide='ignore', invalid='ignore'):
    dd_nsga = (riqueza_nsga - cum_max) / cum_max
    dd_nsga = np.nan_to_num(dd_nsga, nan=0.0)
dd_nsga_pct = dd_nsga * 100

fig_dd = go.Figure()
fig_dd.add_trace(go.Scatter(x=fechas_sim, y=dd_nsga_pct, mode='lines', name='Drawdown NSGA-II', line=dict(color='blue')))
fig_dd.update_layout(
    title=f'Caída Máxima de la Estrategia (Drawdown - Rebalanceo {frecuencia_sel})',
    xaxis_title='Fecha',
    yaxis_title='Drawdown (%)',
    template='plotly_dark',
    hovermode="x unified"
)
fig_dd.show()

## 7. Exportación de Resultados

Guardamos las métricas detalladas en un archivo JSON en los directorios correspondientes para que estén disponibles para los módulos 3 y 4 del optimizador de portafolios. Asimismo, exportamos los pesos del portafolio a un archivo Excel.

In [9]:
metrics_m2 = {
    "tickers": TICKERS_VALIDOS,
    "configuracion": {
        "fecha_inicio": str(FECHA_INICIO),
        "fecha_fin": str(FECHA_FIN),
        "capital_inicial": CAPITAL_INICIAL,
        "frecuencia": frecuencia_sel,
        "lambda_tc": LAMBDA_TC,
        "poblacion": POBLACION,
        "generaciones": GENERACIONES,
        "dias_anio": DIAS_ANIO,
        "rf": RF
    },
    "nsga2_max_sharpe": {
        "retorno": ret_f,
        "riesgo": rsk_f,
        "sharpe_teorico": sharpe_f,
        "pesos": dict(zip(TICKERS_VALIDOS, mejor_pesos.round(6).tolist()))
    },
    "simulacion_riqueza_final": {
        "nsga2": float(riqueza_nsga[-1])
    },
    "trayectorias": {
        "fechas": [d.strftime("%Y-%m-%d") for d in fechas_sim],
        "nsga2": riqueza_nsga.tolist()
    }
}

# Guardar en raíz (para app.py / páginas Streamlit)
with open("resultados_m2.json", "w", encoding="utf-8") as f:
    json.dump(metrics_m2, f, ensure_ascii=False, indent=2)

# Guardar en salidas/ (para otros notebooks)
with open("salidas/resultados_m2.json", "w", encoding="utf-8") as f:
    json.dump(metrics_m2, f, ensure_ascii=False, indent=2)

print("Archivo resultados_m2.json exportado correctamente en ambos directorios.")

# Exportar a Excel
df_export = pd.DataFrame({'Ticker': TICKERS_VALIDOS, 'Peso (%)': (mejor_pesos * 100).round(2)})
excel_path = "salidas/pesos_nsga2.xlsx"
df_export.to_excel(excel_path, index=False, sheet_name='Pesos_NSGA2')
print(f"Pesos óptimos evolutivos exportados correctamente a {excel_path}.")

Archivo resultados_m2.json exportado correctamente en ambos directorios.
Pesos óptimos evolutivos exportados correctamente a salidas/pesos_nsga2.xlsx.
